# **Retreival part**

This part is dedicated to experience retreival part and analysing multiple approaches of chunking and searching for relevant documents.

## **Chunking**

In this part, we are going to do an experience on different chunking methods :

- Fixed length, overlap : basic chunking
- Chunking based on paragraphs

In [11]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from Dataset.upload import get_data
from typing import Literal
import pandas as pd

In [12]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))

PyTorch: 2.14.1+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU memory (GB): 7.96


### **Basic Chunk (simple, paragraph)**

In [13]:
def chunk_characters(text : str, overlap : int, chunk_size : int, chunk_type : Literal["recursive", "simple"] = "simple") -> list : 
    
    if chunk_type == "simple" : 
        splitter = CharacterTextSplitter(
            separator="",
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )
    elif chunk_type == "recursive" : 
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
            length_function=len,
        )

    chunks = splitter.split_text(text)

    return chunks

In [14]:
cour_d_appel = await get_data(juridication="cour_d_appel")

In [15]:
cour_d_appel

,id,themes,text
0,6253c84cbd3db21cbdd84d1c,"[protection des consommateurs, surendettement,...",COUR D'APPEL D'ANGERS \n\nChambre des \n\nREDR...
1,6253c84cbd3db21cbdd84d1d,[bail a loyer (loi du 6 juillet 1989)],Par acte sous seing privé en date du 6 août 19...
2,6253c84cbd3db21cbdd84d1f,"[conventions internationales, accords et conve...",FAITS ET PROCEDURE : La Société SCHLUMBERGER I...
3,6253c84cbd3db21cbdd84d20,"[procedures civiles d'execution, mesures d'exé...","Dans ses écrits devant la Cour, Monsieur X... ..."
4,6253c84cbd3db21cbdd84d21,"[bail a loyer (loi du 6 juillet 1989), preneur...","Par acte sous seing privé du 15 mars 1990, la ..."
...,...,...,...
95,6253c84abd3db21cbdd84ccd,"[contrat de travail, rupture, licenciement, ca...",Monsieur Jacky LE X... a interjeté appel d'un ...
96,6253c84abd3db21cbdd84ccf,"[contrat de travail, execution, modification, ...",Monsieur X... a relevé appel d'un jugement con...
97,6253c84abd3db21cbdd84cd0,"[contrat de travail, execution, salaire, egali...",Madame Jocelyne X... a interjeté appel d'un ju...
98,6253c84abd3db21cbdd84cce,"[contrat de travail, duree determinee, cas de ...",La SNC BANQUETING a relevé appel d'un jugement...


### **Collection handling with chromaDB**

In [16]:
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [33]:
def clear_collection(collection: chromadb.Collection) -> None:
    """Remove existing records from a collection."""
    collection_count = collection.count()

    if collection_count > 0:
        all_ids = collection.get(include=[])["ids"]
        if all_ids:
           collection.delete(ids=all_ids)
        print(f"Cleared {collection_count} records from '{collection.name}'.")
    else:
        print(f"Collection '{collection.name}' is already empty.")

In [23]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=device,
)

cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1825.23it/s]


In [ ]:
DB_PATH = Path(".").resolve().parent / "Collections"
# # A multilingual baseline suitable for French text
# EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# # Load the model once, rather than once per upload
# embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)


print(f"Embedding model running on: {device}")

def create_documents(
    data : pd.DataFrame,
    chunk_type: Literal["recursive", "simple"] = "simple",
    chunk_size: int = 300,
    overlap: int = 100,) -> list :
    
    ids: list[str] = []
    documents: list[str] = []
    metadatas: list[dict] = []

    # Build records from the source DataFrame
    for row in data.itertuples(index=False):
        if pd.isna(row.text) or not str(row.text).strip():
            continue

        source_id = str(row.id)
        source_text = str(row.text)

        source_themes = row.themes
        if isinstance(source_themes, (list, tuple, set)):
            themes_text = ", ".join(map(str, source_themes))
        elif source_themes is None or (
            isinstance(source_themes, float) and pd.isna(source_themes)
        ):
            themes_text = ""
        else:
            themes_text = str(source_themes)

        chunks = chunk_characters(
            text=source_text,
            overlap=overlap,
            chunk_size=chunk_size,
            chunk_type=chunk_type,
        )

        for chunk_index, chunk in enumerate(chunks):
            chunk = str(chunk).strip()

            if not chunk:
                continue

            # Zero-based chunk indices
            chunk_id = f"{source_id}_chunk_{chunk_index}"

            ids.append(chunk_id)
            documents.append(chunk)
            metadatas.append(
                {
                    "source_id": source_id,
                    "chunk_index": chunk_index,
                    "themes": themes_text,
                    "chunk_type": chunk_type,
                }
            )
    return documents, ids, metadatas

def embed_docs(documents : list, batch_size : int, ids : list, metadatas : list, collection : chromadb.Collection)-> bool:
    try : 
        for start in range(0, len(documents), batch_size):
                end = start + batch_size

                batch_documents = documents[start:end]
                batch_ids = ids[start:end]
                batch_metadatas = metadatas[start:end]

                embeddings = embedding_model.encode(
                    batch_documents,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    show_progress_bar=True,
                    batch_size=batch_size,
                )

                collection.add(
                    ids=batch_ids,
                    documents=batch_documents,
                    metadatas=batch_metadatas,
                    embeddings=embeddings.tolist(),
                )      
        print("Added to the collection")
        return True

    except Exception : 
        return False
    
def upload_to_collection(
    configuration : Literal["cosine","ip","l2"],
    data_name : str,
    data: pd.DataFrame,
    chunk_type: Literal["recursive", "simple"] = "simple",
    chunk_size: int = 300,
    overlap: int = 100,
    batch_size: int = 64,
) -> chromadb.Collection:
    """Chunk source documents and store them in a persistent ChromaDB collection."""

    if chunk_type not in ("simple", "recursive"):
        raise ValueError("chunk_type must be 'simple' or 'recursive'.")

    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Require chunk_size > 0 and 0 <= overlap < chunk_size.")

    if batch_size <= 0:
        raise ValueError("batch_size must be positive.")

    required_columns = {"id", "themes", "text"}
    missing_columns = required_columns - set(data.columns)

    if missing_columns:
        raise ValueError(f"Missing DataFrame columns: {missing_columns}")

    collection_name = f"{chunk_type}_{data_name}_{configuration}_jurisprudence"

    # Reuse one persistent database directory
    DB_PATH.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(DB_PATH))

    # Get or create the collection.
    # Embeddings are supplied explicitly, so no embedding_function is needed.
    collection = client.get_or_create_collection(
        name=collection_name,
        configuration={
            {"hnsw": {"space": configuration}}
        }
    )

    # This function intentionally rebuilds the collection on every upload.
    clear_collection(collection)

    documents, ids, metadatas = create_documents(data = data, chunk_type=chunk_type, chunk_size=chunk_size, overlap=overlap)

    if not documents:
        print("No non-empty chunks were generated.")
        return collection

    # Embed and upload records in batches
    result = embed_docs(documents=documents, batch_size=batch_size, ids=ids, metadatas=metadatas, collection=collection)

    if result : 
        print(
            f"Uploaded {collection.count()} chunks to '{collection.name}'."
        )

        return collection


Embedding model running on: cuda


### **Search Functions**

In [ ]:
def KeywordSearch_TFIDF(query : str, docs) -> list :
    return []

def KeywordSearch_BM25(query : str) -> list :
    return []

def SemanticSearch(query : str, collection : chromadb.Collection) -> list :
    return []

### **Defining the metrics functions**